In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests
from scipy.stats import spearmanr
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
median_stats_data = pd.read_csv("../summary_stats_combined/FAC_by_MAR_median_matrix.csv", index_col=0) # read median data

In [ ]:
def get_condition(data_with_conditions, row_name='Diet'):
    '''
    Extract diet information from the metadata file.
    
    Inputs:
        data_with_conditions : str
            Path to the metadata CSV file containing 'Diet' and 'OmicsEarTag' rows.
        row : int (unused, kept for backward compatibility)
            Placeholder parameter (can be ignored).
    
    Outputs:
        pd.DataFrame - Clean mapping of FAC_ID to Diet
    '''
    # Load metadata
    meta = pd.read_csv(data_with_conditions, index_col=0)
    
    # Extract relevant rows 
    diet_row = meta.loc[row_name]
    fac_row = meta.loc['OmicsEarTag']
    
    # Build mapping
    diet_map = pd.DataFrame({
        'FAC_ID': fac_row.values,
        row_name: diet_row.values
    })
    
    # Drop unwanted rows (header-like entries)
    diet_map = diet_map[~diet_map['FAC_ID'].isin(['Orig_ID', 'IDFemaleAgingColony'])].reset_index(drop=True)
    
    return diet_map

In [ ]:
diet_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv") # get diet data

In [ ]:
df_merged_diet = median_stats_data.merge( # make the dataframe
    diet_data,
    left_on="FAC",
    right_on="FAC_ID",
    how="inner"
)

In [ ]:
df_merged_diet = df_merged_diet.drop(columns=['FAC_ID']) # remove double columns

In [ ]:
df_merged_diet["Diet_num"] = df_merged_diet["Diet"].map({"HF": 1, "CD": 0}) # code HF as 1, and CD as 0

# Spearman rank

In [ ]:
def compute_spearman_diet_correlation(df_merged_diet):
    """
    Compute Spearman correlation between MAR features and diet.
    Inputs:
        df_merged_diet : pd.DataFrame
            DataFrame containing MAR features and diet information.
    Outputs:
        pd.DataFrame - Spearman correlation results with FDR correction.
    """

    # Automatically select all columns starting with "MAR"
    mar_cols = [c for c in df_merged_diet.columns if c.startswith("MAR")]

    results = []

    # Loop through each MAR feature
    for col in mar_cols:
        # Keep only relevant columns and drop rows with missing values
        subset = df_merged_diet[[col, "Diet_num"]].dropna()
        
        # Ensure the feature has variability (required for correlation)
        if subset[col].nunique() > 1:
            # Compute Spearman rank correlation (non-parametric)
            rho, pval = spearmanr(subset[col], subset["Diet_num"])
            results.append([col, rho, pval])

    # Convert results to DataFrame
    df_spearman = pd.DataFrame(
        results,
        columns=["Feature", "Spearman_rho", "p_value"]
    )

    # Apply multiple testing correction (Benjamini-Hochberg FDR)
    df_spearman["FDR"] = multipletests(
        df_spearman["p_value"],
        method="fdr_bh"
    )[1]

    # Filter significant features (FDR < 0.05) and sort by correlation strength
    sig_features = df_spearman[df_spearman["FDR"] < 0.05] \
        .sort_values("Spearman_rho", ascending=False)

    # Preview top results (not returned, just for quick inspection)
    sig_features.head(10)

    # Save results to disk
    sig_features.to_csv("../final_result/diet_spearman_results.csv", index=False)
    df_spearman.to_csv("../final_result/diet_spearman_results_all.csv", index=False)

    # Return full results and significant subset
    return df_spearman, sig_features

In [ ]:
# Compute spearman correlation for diet
df_spearman_diet, sig_features_diet = compute_spearman_diet_correlation(df_merged_diet)

# print for a sanity check
top = sig_features_diet.head(10)["Feature"]

# sanity check with the plot to see if diets where coded correctly
sns.boxplot(
    data=df_merged_diet,
    x="Diet",
    y=top.iloc[0]
)
plt.title(top.iloc[0])
plt.show()